# Experiment 2 — Manipulation Technique Classification

*Uses the same data, template groups, and train/test split as Experiment 1 (`src/data.py`).*

The second experiment predicts **which technique** a manipulative text uses.

Mathur et al. annotate every dark pattern with a *Pattern Type*. We map the types to our technique labels:

| Our label | Mathur pattern types | Example |
|---|---|---|
| `urgency` | Countdown Timer, Limited-time Message | *"Sale ends in 02:14:55"* |
| `scarcity` | Low-stock Message, High-demand Message | *"Only 2 left in stock"* |
| `social_proof` | Activity Notification, Testimonials of Uncertain Origin | *"Anna from Denver just bought this"* |
| `emotional_pressure` | Confirmshaming | *"No thanks, I don't like saving money"* |

Our earlier labels `fear`, `exaggeration`, and `authority` have **no counterpart** in Mathur's taxonomy, so they cannot be trained from this dataset and are dropped here.
About 110 manipulative texts belong to other pattern types (e.g. *Pressured Selling*, *Hard to Cancel*); they stay in the data with all four technique labels set to 0.

**Note:** in this dataset every text has **at most one** technique, so the task is effectively multi-class. We keep the One-vs-Rest multi-label setup so that the code and metrics stay comparable with our original data, where one ad can carry several techniques.

In [ ]:
# 0. Make the shared `src` package importable (works from the repo root or from notebooks/)

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
# 17. Imports for multi-label classification

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    multilabel_confusion_matrix,
    ConfusionMatrixDisplay,
    make_scorer
)

from src.data import RANDOM_STATE, load_all, save_table

MULTILABEL_COLUMNS = [
    "urgency",
    "scarcity",
    "social_proof",
    "emotional_pressure"
]

In [ ]:
# Load data and the shared split (same as Experiment 1)

main_df, groups, real_df, external_synthetic_df, train_idx, test_idx = load_all()

print("Main examples:", len(main_df))
print("Training examples:", len(train_idx), "| Test examples:", len(test_idx))

## 12. Inspect the multi-label distribution

Each individual manipulation technique is much less frequent than the binary `manipulative` class.

This means that every one-vs-rest classifier has many more negative than positive examples.  
For that reason, Logistic Regression is trained with `class_weight="balanced"`.

In [ ]:
# 18. Multi-label distribution in the main dataset

label_counts = main_df[MULTILABEL_COLUMNS].sum().sort_values(ascending=False)

label_distribution = pd.DataFrame({
    "positive_examples": label_counts,
    "negative_examples": len(main_df) - label_counts,
    "positive_rate": label_counts / len(main_df)
})

display(
    label_distribution.style.format({
        "positive_rate": "{:.1%}"
    })
)

plt.figure(figsize=(9, 4))
label_counts.plot(kind="bar")
plt.title("Positive Examples per Manipulation Technique")
plt.xlabel("Technique")
plt.ylabel("Number of positive examples")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 13. Multi-label train/test split

We reuse **the same group-aware `train_idx` / `test_idx`** as in the binary experiment, so near-duplicates stay on one side and all experiments share one test set.
Then we check the number of positive examples for every label in both subsets.

In [ ]:
# 19. Prepare X and Y for the multi-label task

X_multi = main_df["text"]
Y_multi = main_df[MULTILABEL_COLUMNS].astype(int)

X_multi_train, X_multi_test = X_multi.iloc[train_idx], X_multi.iloc[test_idx]
Y_multi_train, Y_multi_test = Y_multi.iloc[train_idx], Y_multi.iloc[test_idx]

split_distribution = pd.DataFrame({
    "train_positive": Y_multi_train.sum(),
    "test_positive": Y_multi_test.sum()
})

display(split_distribution)

## 14. TF-IDF + One-vs-Rest Logistic Regression

`OneVsRestClassifier` trains one binary classifier for each technique.

For example:

- classifier 1: urgency vs. not urgency
- classifier 2: scarcity vs. not scarcity
- ...
- classifier 7: emotional pressure vs. not emotional pressure

All seven predictions can be active at the same time.

In [ ]:
# 20. Build and train the multi-label baseline

multilabel_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2)
        )
    ),
    (
        "classifier",
        OneVsRestClassifier(
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        )
    )
])

multilabel_model.fit(X_multi_train, Y_multi_train)

print("Multi-label model trained.")

## 15. Evaluate the multi-label model

In [ ]:
# 21. Multi-label predictions

Y_multi_pred = multilabel_model.predict(X_multi_test)
Y_multi_prob = multilabel_model.predict_proba(X_multi_test)

multi_metrics = pd.DataFrame({
    "Metric": [
        "Subset accuracy",
        "Hamming loss",
        "Micro Precision",
        "Micro Recall",
        "Micro F1",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Micro Jaccard"
    ],
    "Value": [
        accuracy_score(Y_multi_test, Y_multi_pred),
        hamming_loss(Y_multi_test, Y_multi_pred),
        precision_score(Y_multi_test, Y_multi_pred, average="micro", zero_division=0),
        recall_score(Y_multi_test, Y_multi_pred, average="micro", zero_division=0),
        f1_score(Y_multi_test, Y_multi_pred, average="micro", zero_division=0),
        precision_score(Y_multi_test, Y_multi_pred, average="macro", zero_division=0),
        recall_score(Y_multi_test, Y_multi_pred, average="macro", zero_division=0),
        f1_score(Y_multi_test, Y_multi_pred, average="macro", zero_division=0),
        jaccard_score(Y_multi_test, Y_multi_pred, average="micro", zero_division=0)
    ]
})

display(multi_metrics.style.format({"Value": "{:.3f}"}))

### Meaning of the main multi-label metrics

- **Subset accuracy:** exact match; every one of the seven labels must be predicted correctly for the whole text.
- **Hamming loss:** fraction of individual label decisions that are wrong. Lower is better.
- **Micro F1:** combines all label decisions and is influenced more by common labels.
- **Macro F1:** calculates F1 for each technique separately and then averages them equally.
- **Jaccard score:** measures overlap between the predicted set and true set of labels.

In [ ]:
# 22. Per-label performance

per_label_rows = []

for i, label in enumerate(MULTILABEL_COLUMNS):
    per_label_rows.append({
        "label": label,
        "precision": precision_score(
            Y_multi_test.iloc[:, i],
            Y_multi_pred[:, i],
            zero_division=0
        ),
        "recall": recall_score(
            Y_multi_test.iloc[:, i],
            Y_multi_pred[:, i],
            zero_division=0
        ),
        "f1": f1_score(
            Y_multi_test.iloc[:, i],
            Y_multi_pred[:, i],
            zero_division=0
        ),
        "support": int(Y_multi_test.iloc[:, i].sum())
    })

per_label_metrics = pd.DataFrame(per_label_rows).sort_values(
    "f1",
    ascending=False
)

display(
    per_label_metrics.style.format({
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1": "{:.3f}"
    })
)

## 16. Confusion matrix for each manipulation technique

In [ ]:
# 23. One confusion matrix per label

for i, label in enumerate(MULTILABEL_COLUMNS):
    ConfusionMatrixDisplay.from_predictions(
        Y_multi_test.iloc[:, i],
        Y_multi_pred[:, i],
        display_labels=["No", "Yes"],
        values_format="d"
    )
    plt.title(f"Confusion Matrix — {label}")
    plt.tight_layout()
    plt.show()

## 17. Inspect individual predictions

In [ ]:
# 24. Convert vectors of 0/1 values into readable label lists

def active_labels(values):
    return [
        label
        for label, value in zip(MULTILABEL_COLUMNS, values)
        if int(value) == 1
    ]

multi_prediction_results = pd.DataFrame({
    "text": X_multi_test.reset_index(drop=True),
    "true_labels": [
        active_labels(row)
        for row in Y_multi_test.to_numpy()
    ],
    "predicted_labels": [
        active_labels(row)
        for row in Y_multi_pred
    ]
})

multi_prediction_results["exact_match"] = (
    multi_prediction_results["true_labels"].astype(str)
    == multi_prediction_results["predicted_labels"].astype(str)
)

display(multi_prediction_results.head(20))

## 18. Multi-label error analysis

In [ ]:
# 25. Show examples where the full label set was not predicted correctly

multi_errors = multi_prediction_results[
    ~multi_prediction_results["exact_match"]
].copy()

print("Texts with at least one label error:", len(multi_errors))
display(multi_errors.head(30))

## 19. 5-fold cross-validation for the multi-label baseline

`StratifiedKFold` cannot stratify a multi-label target matrix, so we use **`GroupKFold`**, which keeps near-duplicate templates within one fold.
The TF-IDF + One-vs-Rest pipeline is refitted inside every fold, so no TF-IDF information leaks from the validation fold.

In [ ]:
# 26. Multi-label cross-validation

multi_cv = GroupKFold(n_splits=5)

micro_f1_scorer = make_scorer(
    f1_score,
    average="micro",
    zero_division=0
)

macro_f1_scorer = make_scorer(
    f1_score,
    average="macro",
    zero_division=0
)

multi_cv_results = cross_validate(
    multilabel_model,
    X_multi,
    Y_multi,
    groups=groups,
    cv=multi_cv,
    scoring={
        "micro_f1": micro_f1_scorer,
        "macro_f1": macro_f1_scorer
    }
)

multi_cv_summary = pd.DataFrame({
    "Metric": ["Micro F1", "Macro F1"],
    "Mean": [
        multi_cv_results["test_micro_f1"].mean(),
        multi_cv_results["test_macro_f1"].mean()
    ],
    "Std": [
        multi_cv_results["test_micro_f1"].std(),
        multi_cv_results["test_macro_f1"].std()
    ]
})

display(
    multi_cv_summary.style.format({
        "Mean": "{:.3f}",
        "Std": "{:.3f}"
    })
)

## 20. Multi-label check on our real advertisements

The classifier is retrained on the full main dataset and applied to our real advertising examples, using the four labels both datasets share.

The set is small and comes from a different domain, so this is an exploratory sanity check.

In [ ]:
# 27. Train on the full main dataset

final_multilabel_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2)
        )
    ),
    (
        "classifier",
        OneVsRestClassifier(
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        )
    )
])

final_multilabel_model.fit(
    main_df["text"],
    main_df[MULTILABEL_COLUMNS]
)

print("Final multi-label model trained on the full main dataset.")

In [ ]:
# 28. Evaluate on the real examples

if len(real_df) > 0:
    real_multi_true = real_df[MULTILABEL_COLUMNS].astype(int)
    real_multi_pred = final_multilabel_model.predict(real_df["text"])
    real_multi_prob = final_multilabel_model.predict_proba(real_df["text"])

    real_multi_results = pd.DataFrame({
        "text": real_df["text"].reset_index(drop=True),
        "true_labels": [
            active_labels(row)
            for row in real_multi_true.to_numpy()
        ],
        "predicted_labels": [
            active_labels(row)
            for row in real_multi_pred
        ]
    })

    real_multi_results["exact_match"] = (
        real_multi_results["true_labels"].astype(str)
        == real_multi_results["predicted_labels"].astype(str)
    )

    print(
        "Real multi-label Micro F1:",
        round(
            f1_score(
                real_multi_true,
                real_multi_pred,
                average="micro",
                zero_division=0
            ),
            3
        )
    )

    print(
        "Real multi-label Macro F1:",
        round(
            f1_score(
                real_multi_true,
                real_multi_pred,
                average="macro",
                zero_division=0
            ),
            3
        )
    )

    display(real_multi_results)
else:
    print("No real examples were found.")

## 21. Probability inspection for real advertisements

In [ ]:
# 29. Show predicted probabilities for every technique

if len(real_df) > 0:
    probability_table = pd.DataFrame(
        real_multi_prob,
        columns=MULTILABEL_COLUMNS
    )

    probability_table.insert(
        0,
        "text",
        real_df["text"].reset_index(drop=True)
    )

    display(
        probability_table.style.format({
            label: "{:.3f}"
            for label in MULTILABEL_COLUMNS
        })
    )

## 22. What the multi-label TF-IDF classifiers learned

Because each technique has its own Logistic Regression classifier, we can inspect the terms with the largest positive coefficients for every label.

This helps us determine whether the model is learning plausible persuasion cues or merely memorizing artificial wording patterns.

In [ ]:
# 30. Top TF-IDF features for every manipulation technique

multi_tfidf = final_multilabel_model.named_steps["tfidf"]
multi_classifier = final_multilabel_model.named_steps["classifier"]

multi_feature_names = np.array(
    multi_tfidf.get_feature_names_out()
)

top_n = 10
feature_tables = []

for label, estimator in zip(
    MULTILABEL_COLUMNS,
    multi_classifier.estimators_
):
    coefficients = estimator.coef_[0]
    top_idx = np.argsort(coefficients)[-top_n:][::-1]

    table = pd.DataFrame({
        "label": label,
        "feature": multi_feature_names[top_idx],
        "coefficient": coefficients[top_idx]
    })

    feature_tables.append(table)

top_features_by_label = pd.concat(
    feature_tables,
    ignore_index=True
)

display(top_features_by_label)

In [ ]:
# 31. Save results for the comparison in Experiments 3 and 4

save_table(multi_metrics, "exp2_test_metrics")
save_table(multi_cv_summary, "exp2_cv_summary")
print("Saved to results/.")

# Experiment 2 — Summary

### Model
- **Representation:** TF-IDF unigrams + bigrams
- **Classifier:** One-vs-Rest Logistic Regression
- **Class balancing:** `class_weight="balanced"`
- **Output:** four technique labels (`urgency`, `scarcity`, `social_proof`, `emotional_pressure`)

### Current limitations
- `fear`, `exaggeration`, and `authority` cannot be learned from this dataset.
- Every training text has at most one technique, so co-occurring techniques are not represented.
- Technique cues in website UI text are highly templated (timers, stock counters), which makes them easy for lexical models.
- TF-IDF mainly captures lexical patterns rather than contextual meaning.

### Next experiment
Replace the sparse TF-IDF representation with **dense embeddings** (Word2Vec, GloVe) and compare.